1. Create a source (Bronze) table that is continuously updated.  
2. Turn on CDF for that table.  
3. Read only the incremental changes (inserts, updates, deletes).  
4. MERGE those changes into a downstream (Silver) table.

In [0]:
# Sample initial data
initial = spark.createDataFrame(
    [(1, "Widget A", 10.00),
     (2, "Widget B", 12.50)],
    ["id", "name", "price"]
)
initial.show()

+---+--------+-----+
| id|    name|price|
+---+--------+-----+
|  1|Widget A| 10.0|
|  2|Widget B| 12.5|
+---+--------+-----+



In [0]:
initial.printSchema()

root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- price: double (nullable = true)



In [0]:
df_pandas= initial.toPandas()

In [0]:
df_pandas.to_csv("/Volumes/main/demo_schema/demo_vol/base_data/scp_test.csv",header=True)

In [0]:
from pyspark.sql.functions import current_timestamp

In [0]:
spark.sql("Drop table main.demo_schema.scd2_table_demo")

DataFrame[]

In [0]:
spark.sql("""
          Create table main.demo_schema.scd2_table_demo 
           (
          id int,
          name string,
          price double,
          effective_date timestamp,
          end_date timestamp
          ) """)

DataFrame[]

In [0]:
spark.read.table("main.demo_schema.scd2_table_demo").display()

id,name,price,effective_date,end_date


In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import lit,current_timestamp
from pyspark.sql.types import StringType,TimestampType,IntegerType,DoubleType
dt= DeltaTable.forName(spark,"main.demo_schema.scd2_table_demo")


In [0]:
# update Current row
dt.alias("t").\
    merge(
        source=initial.alias("s"),
        condition="s.id=t.id and t.end_date is null"
    ).\
        whenMatchedUpdate(
            set ={"t.end_date":current_timestamp()}
                          ).\
                        execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
initial=initial.withColumn("effective_date",current_timestamp()).withColumn("end_date",lit(None).cast(TimestampType())).select(initial.id.cast(IntegerType()),initial.name.cast(StringType()),initial.price.cast(DoubleType()),"effective_date","end_date").write.mode("append").saveAsTable("main.demo_schema.scd2_table_demo")


In [0]:
spark.read.table("main.demo_schema.scd2_table_demo").display()

id,name,price,effective_date,end_date
1,Widget A,10.0,2026-01-08T08:28:42.227Z,null
2,Widget B,12.5,2026-01-08T08:28:42.227Z,null


In [0]:
# Sample initial data
initial = spark.createDataFrame(
    [
     (2, "Widget B",15)],
    ["id", "name", "price"]
)
initial.show()

+---+--------+-----+
| id|    name|price|
+---+--------+-----+
|  2|Widget B|   15|
+---+--------+-----+



In [0]:
# update Current row
dt.alias("t").\
    merge(
        source=initial.alias("s"),
        condition="s.id=t.id and t.end_date is null"
    ).\
        whenMatchedUpdate(
            set ={"t.end_date":current_timestamp()}
                          ).\
                        execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.read.table("main.demo_schema.scd2_table_demo").display()

id,name,price,effective_date,end_date
2,Widget B,12.5,2026-01-08T08:28:42.227Z,2026-01-08T08:28:50.969Z
1,Widget A,10.0,2026-01-08T08:28:42.227Z,null


In [0]:
initial=initial.withColumn("effective_date",current_timestamp()).withColumn("end_date",lit(None).cast(TimestampType())).select(initial.id.cast(IntegerType()),initial.name.cast(StringType()),initial.price.cast(DoubleType()),"effective_date","end_date").write.mode("append").saveAsTable("main.demo_schema.scd2_table_demo")

In [0]:
spark.read.table("main.demo_schema.scd2_table_demo").display()

id,name,price,effective_date,end_date
1,Widget A,10.0,2026-01-08T08:28:42.227Z,null
2,Widget B,12.5,2026-01-08T08:28:42.227Z,2026-01-08T08:28:50.969Z
2,Widget B,15.0,2026-01-08T08:29:03.064Z,null


In [0]:
# Sample initial data
initial = spark.createDataFrame(
    [
     (1, "Widget BA",15)],
    ["id", "name", "price"]
)
initial.show()

+---+---------+-----+
| id|     name|price|
+---+---------+-----+
|  1|Widget BA|   15|
+---+---------+-----+



In [0]:
# update Current row
dt.alias("t").\
    merge(
        source=initial.alias("s"),
        condition="s.id=t.id and t.end_date is null"
    ).\
        whenMatchedUpdate(
            set ={"t.end_date":current_timestamp(),
                   "t.name":"s.name",
                    "t.price":"s.price"}
                          ).\
                        execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.read.table("main.demo_schema.scd2_table_demo").display()

id,name,price,effective_date,end_date
2,Widget B,12.5,2026-01-08T08:28:42.227Z,2026-01-08T08:28:50.969Z
2,Widget B,15.0,2026-01-08T08:29:03.064Z,null
1,Widget BA,15.0,2026-01-08T08:28:42.227Z,2026-01-08T08:29:06.747Z


In [0]:
initial.withColumn("effective_date",current_timestamp()).withColumn("end_date",lit(None).cast(TimestampType())).select(initial.id.cast(IntegerType()),initial.name.cast(StringType()),initial.price.cast(DoubleType()),"effective_date","end_date").write.mode("append").saveAsTable("main.demo_schema.scd2_table_demo")

In [0]:
spark.read.table("main.demo_schema.scd2_table_demo").display()

id,name,price,effective_date,end_date
2,Widget B,12.5,2026-01-08T08:28:42.227Z,2026-01-08T08:28:50.969Z
2,Widget B,15.0,2026-01-08T08:29:03.064Z,null
1,Widget BA,15.0,2026-01-08T08:28:42.227Z,2026-01-08T08:29:06.747Z
1,Widget BA,15.0,2026-01-08T08:32:28.127Z,null


In [0]:
spark.read.table("main.demo_schema.scd2_table_demo").explain()

== Physical Plan ==
*(1) ColumnarToRow
+- PhotonResultStage
   +- PhotonScan parquet main.demo_schema.scd2_table_demo[id#15735,name#15736,price#15737,effective_date#15738,end_date#15739] DataFilters: [], DictionaryFilters: [], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[s3://dbstorage-prod-lw6mu/uc/f8d04c04-4088-4d2a-9975-0080fdf78f74..., OptionalDataFilters: [], PartitionFilters: [], ReadSchema: struct<id:int,name:string,price:double,effective_date:timestamp,end_date:timestamp>, RequiredDataFilters: []


== Photon Explanation ==
The query is fully supported by Photon.
== Optimizer Statistics (table names per statistics state) ==
  missing = 
  partial = 
  full    = scd2_table_demo

